# 🏛️ LegisSum — End-to-End Demo
**Fine-tuning Mistral-7B on BillSum with QLoRA**

Run this in Google Colab (free T4 GPU) or locally.

---

In [ ]:
# Step 0: Install dependencies
!pip install -q datasets transformers peft trl bitsandbytes accelerate rouge-score

In [ ]:
# Step 1: Download & preview BillSum
from datasets import load_dataset

ds = load_dataset('FiscalNote/billsum')
sample = ds['train'][42]

print('📄 TITLE:', sample['title'])
print()
print('📜 BILL TEXT (first 500 chars):')
print(sample['text'][:500])
print()
print('✅ OFFICIAL SUMMARY:')
print(sample['summary'])

In [ ]:
# Step 2: Preprocess — filter and format
import sys, os
sys.path.insert(0, '../src')

# Run the preprocess script
!python ../src/01_download_data.py
!python ../src/02_preprocess.py

In [ ]:
# Step 3: Train (QLoRA fine-tuning)
# ⚠️ This takes ~3-4 hours on Colab T4 for 2 epochs
# Reduce EPOCHS in 03_train.py to 1 for a quicker demo
!python ../src/03_train.py

In [ ]:
# Step 4: Before/After evaluation
!python ../src/04_evaluate.py

In [ ]:
# Step 5: View results
import json
with open('../outputs/rouge_scores.json') as f:
    scores = json.load(f)

print('📊 ROUGE Scores')
print(f'{"Metric":<10} {"Base":>10} {"Fine-tuned":>12} {"Δ":>8}')
print('-' * 42)
for k in ['rouge1', 'rouge2', 'rougeL']:
    b = scores['base'][k]
    f = scores['finetuned'][k]
    print(f'{k:<10} {b:>10.4f} {f:>12.4f} {f-b:>+8.4f}')